# Seven-Day Meter

CE 663 · Day-ahead electricity forecasting for previously unseen buildings

Can a small pooled model using 168 hourly kWh readings and the next day's calendar beat copying yesterday or the same day last week? This experiment uses the four BDG2 sites named by BuildingsBench and applies its own eligibility rules to the authors' cleaned meter export. The published benchmark reports a processed subset of 611 buildings. The counts below refer to the actual file and the rules used here.

Run with Python 3, NumPy and pandas. The included CSV files are read from the package’s data folder. Open the notebook from the notebooks folder or run it from the package root. The saved outputs contain the original results. docs/QUICKSTART.md explains the command-line runner.

[BDG2 cleaned electricity export](https://github.com/buds-lab/building-data-genome-project-2/blob/master/data/meters/cleaned/electricity_cleaned.csv) · [BDG2 data paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC7591488/) · [BuildingsBench paper](https://proceedings.neurips.cc/paper_files/paper/2023/file/3f17bf868966df01ca125e5bbc9ee24e-Paper-Datasets_and_Benchmarks.pdf)

## Protocol

Use each published local-clock hour label as delivered in the cleaned file. Forecast once per day at midnight, with both the 168-hour lookback and the 24-hour target inside the same calendar year. Keep windows with 192 consecutive hour labels, finite nonnegative readings, and a positive mean across the lookback. The source uses a regular local-clock grid, so this experiment evaluates that grid as published. The upload contract also uses local-clock labels and requires a 24-hour day.

Within each site, order building IDs by a fixed salted SHA-256 hash and reserve roughly 60% for fitting, 20% for tuning, and 20% for testing. Use only 2016 dates for fitting and tuning and only 2017 dates for testing. Require 30 eligible forecast days per building in its assigned year. The cutoff avoids reporting a building's typical error from a handful of days. Buildings in different splits can share a site. Use identical windows for all methods.

Train ridge regression on normalized hourly readings. Divide each input and target by that window's observed 168-hour mean. Use the seven same-hour historical readings, the mean over the final 24 observed hours, and next-day calendar features. Weight each fitting building equally. Tune the ridge penalty and select the default method by lowest median per-building normalized error on validation. Score the frozen methods on 2017, with building-level uncertainty and weekday/weekend summaries. Test-day totals and readings enter only the reported metrics, never the forecast.

In [1]:
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd

sites = ('Bear', 'Fox', 'Panther', 'Rat')
files = [folder / name for folder in (Path.cwd() / 'data', Path.cwd().parent / 'data')
         for name in ('electricity_cleaned.csv', 'electricity_cleaned.txt')]
source = next((path for path in files if path.is_file() and path.stat().st_size > 1_000_000), None)
if source is None:
    raise FileNotFoundError('Download the full BDG2 cleaned electricity CSV as described above.')
with source.open('rb') as stream:
    digest = sha256()
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        digest.update(chunk)
print('Source SHA-256:', digest.hexdigest())

meter = pd.read_csv(source, usecols=lambda name: name == 'timestamp' or name.partition('_')[0] in sites)
times = pd.DatetimeIndex(pd.to_datetime(meter.pop('timestamp'), format='%Y-%m-%d %H:%M:%S', errors='raise'))
if times.has_duplicates or not times.is_monotonic_increasing or not times.equals(times.floor('h')):
    raise ValueError('Expected unique, sorted hour-aligned timestamps in the published export.')
readings = meter.to_numpy(dtype=float).T.copy()
building_ids = meter.columns.to_list()
del meter
site_by_id = {name: name.partition('_')[0] for name in building_ids}
print(f'File: {len(times):,} timestamp rows and {len(building_ids)} columns at the four sites')
print(pd.Series(site_by_id).value_counts().sort_index().to_string())


def timestamp_windows(year):
    possible = np.flatnonzero((times.year == year) & (times.hour == 0))
    starts = possible[(possible >= 168) & (possible + 23 < len(times))]
    elapsed = times[starts + 23] - times[starts - 168]
    return starts[(elapsed == pd.Timedelta(hours=191)) & (times[starts - 168].year == year)
                  & (times[starts + 23].year == year)]


starts_by_year = {year: timestamp_windows(year) for year in (2016, 2017)}
valid_hours = np.isfinite(readings) & (readings >= 0)
hour_counts = np.pad(np.cumsum(valid_hours, axis=1, dtype=np.int32), ((0, 0), (1, 0)))
positive_counts = np.pad(np.cumsum(valid_hours & (readings > 0), axis=1, dtype=np.int32), ((0, 0), (1, 0)))
valid_days = {}
for year, starts in starts_by_year.items():
    complete = (hour_counts[:, starts + 24] - hour_counts[:, starts - 168]) == 192
    positive_history = (positive_counts[:, starts] - positive_counts[:, starts - 168]) > 0
    valid_days[year] = complete & positive_history
del hour_counts, positive_counts, valid_hours


def stable_order(name):
    return sha256(('ce663-seven-day-meter-v1:' + name).encode()).hexdigest()


assigned = {}
for site in sites:
    group = sorted((i for i, name in enumerate(building_ids) if site_by_id[name] == site),
                   key=lambda i: stable_order(building_ids[i]))
    first, second = round(.60 * len(group)), round(.80 * len(group))
    assigned.update({i: 'train' for i in group[:first]})
    assigned.update({i: 'validation' for i in group[first:second]})
    assigned.update({i: 'test' for i in group[second:]})
years = {'train': 2016, 'validation': 2016, 'test': 2017}
splits = {split: sorted(i for i in assigned if assigned[i] == split and valid_days[year][i].sum() >= 30)
          for split, year in years.items()}
assert sum(map(len, splits.values())) == len(set().union(*map(set, splits.values())))
assert all(splits.values())

coverage_rows = []
for site in sites:
    indices = [i for i, name in enumerate(building_ids) if site_by_id[name] == site]
    for year in (2016, 2017):
        counts = valid_days[year][indices].sum(axis=1)
        coverage_rows.append((site, year, len(indices), int((counts > 0).sum()), int((counts >= 30).sum()), int(counts.sum())))
coverage = pd.DataFrame(coverage_rows, columns=['Site', 'Year', 'CSV columns', 'Any complete days', '30+ complete days', 'Complete days'])
split_rows = [(split, years[split], sum(s == split for s in assigned.values()), len(indices),
               int(valid_days[years[split]][indices].sum())) for split, indices in splits.items()]
split_table = pd.DataFrame(split_rows, columns=['Split', 'Year', 'Assigned', 'Eligible buildings', 'Forecast days'])
print('\nCoverage before split assignment:')
print(coverage.to_string(index=False))
print('\nFinal evaluation cohorts:')
print(split_table.to_string(index=False))
print('\nEligible buildings by site and split:')
print(pd.DataFrame({split: pd.Series([site_by_id[building_ids[i]] for i in indices]).value_counts()
                    for split, indices in splits.items()}).reindex(sites).fillna(0).astype(int).to_string())

Source SHA-256: b6ffc9b4dfcefe5c753594730a08ae822b0d50fec6815abb8f185591e6c630a3
File: 17,544 timestamp rows and 639 columns at the four sites
Bear        92
Fox        137
Panther    105
Rat        305

Coverage before split assignment:
   Site  Year  CSV columns  Any complete days  30+ complete days  Complete days
   Bear  2016           92                 90                 89          27169
   Bear  2017           92                 91                 90          28172
    Fox  2016          137                136                135          43962
    Fox  2017          137                136                134          45065
Panther  2016          105                104                103          21906
Panther  2017          105                105                104          34369
    Rat  2016          305                304                303          94888
    Rat  2017          305                281                277          84914

Final evaluation cohorts:
     Split  Yea

In [2]:
methods = ['Copy yesterday', 'Copy last week', 'Pooled ridge']
hour = np.arange(24)
hours_in_week = 168
lag_offsets = -24 * np.arange(1, 8)
alphas = (0.0001, 0.001, 0.01, 0.1)


def design(history, weekdays):
    # A history row is oldest to newest, with hour zero at the oldest midnight.
    scales = history.mean(axis=1)
    lags = history[:, hours_in_week + hour[:, None] + lag_offsets[None, :]] / scales[:, None, None]
    latest_day_mean = history[:, -24:].mean(axis=1) / scales
    n = len(history)
    return np.concatenate([
        np.ones((n, 24, 1)), lags, np.broadcast_to(latest_day_mean[:, None, None], (n, 24, 1)),
        np.broadcast_to(np.eye(24)[None, :, 1:], (n, 24, 23)),
        np.broadcast_to(np.eye(7)[weekdays, None, 1:], (n, 24, 6)),
    ], axis=2), scales


def windows(index, year):
    starts = starts_by_year[year][valid_days[year][index]]
    history = readings[index, starts[:, None] + np.arange(-168, 0)]
    actual = readings[index, starts[:, None] + hour]
    weekdays = times[starts].dayofweek.to_numpy()
    return starts, history, actual, weekdays


gram = np.zeros((38, 38))
rhs = np.zeros(38)
for index in splits['train']:
    _, history, actual, weekdays = windows(index, 2016)
    features, scales = design(history, weekdays)
    x = features.reshape(-1, features.shape[-1])
    target = (actual / scales[:, None]).ravel()
    gram += (x.T @ x) / len(x)
    rhs += (x.T @ target) / len(x)
penalty = np.eye(len(rhs))
penalty[0, 0] = 0
weights_by_alpha = np.linalg.solve(gram + np.asarray(alphas)[:, None, None] * len(splits['train']) * penalty,
                                  np.broadcast_to(rhs, (len(alphas), len(rhs)))[:, :, None]).squeeze(-1)


def nmae(errors, actual):
    # Aggregate within a building before taking a median over buildings.
    return np.sum(errors, axis=(0, 1)) / np.sum(actual)


val_errors = []
for index in splits['validation']:
    _, history, actual, weekdays = windows(index, 2016)
    features, scales = design(history, weekdays)
    ridge = np.maximum(0, (features @ weights_by_alpha.T) * scales[:, None, None])
    predictions = np.concatenate([history[:, -24:, None], history[:, :24, None], ridge], axis=2)
    val_errors.append(nmae(np.abs(predictions - actual[:, :, None]), actual))

validation_medians = np.median(val_errors, axis=0)
best_alpha_index = int(np.argmin(validation_medians[2:]))
weights = weights_by_alpha[best_alpha_index]
validation_scores = validation_medians[[0, 1, 2 + best_alpha_index]]
default = int(np.argmin(validation_scores))
best_baseline = int(np.argmin(validation_scores[:2]))
print('Ridge penalties and validation median normalized errors:')
print(dict(zip(alphas, (100 * validation_medians[2:]).round(3).tolist())))
print('Validation median normalized error (%):', dict(zip(methods, (100 * validation_scores).round(3).tolist())))
print('Default frozen before scoring test:', methods[default])

Ridge penalties and validation median normalized errors:
{0.0001: 10.476, 0.001: 10.429, 0.01: 10.397, 0.1: 11.877}
Validation median normalized error (%): {'Copy yesterday': 11.4, 'Copy last week': 14.147, 'Pooled ridge': 10.397}
Default frozen before scoring test: Pooled ridge


In [3]:
results = []
day_results = []
for index in splits['test']:
    _, history, actual, weekdays = windows(index, 2017)
    features, scales = design(history, weekdays)
    prediction = np.stack([history[:, -24:], history[:, :24], np.maximum(0, (features @ weights) * scales[:, None])], axis=2)
    errors = np.abs(prediction - actual[:, :, None])
    site = site_by_id[building_ids[index]]
    totals = errors.sum(axis=(0, 1))
    results.append((site, len(history) * 24, totals, nmae(errors, actual)))
    for day_type, mask in [('Weekday', weekdays < 5), ('Weekend', weekdays >= 5)]:
        if mask.any() and actual[mask].sum() > 0:
            day_results.append((day_type, nmae(errors[mask], actual[mask])))

test_nmae = np.stack([row[3] for row in results])
test_median = np.median(test_nmae, axis=0)
test_mae = np.sum([row[2] for row in results], axis=0) / sum(row[1] for row in results)
metrics = pd.DataFrame({'Method': methods, '2016 validation median NMAE %': 100 * validation_scores,
                        '2017 test hourly MAE kWh': test_mae, '2017 test median building NMAE %': 100 * test_median})
print(metrics.round(3).to_string(index=False))
print('Site median building NMAE (%):')
for site in sites:
    mask = np.asarray([row[0] == site for row in results])
    if mask.any():
        print(site, int(mask.sum()), (100 * np.median(test_nmae[mask], axis=0)).round(2).tolist())
for day_type in ('Weekday', 'Weekend'):
    group = [values for name, values in day_results if name == day_type]
    if group:
        print(day_type, len(group), (100 * np.median(group, axis=0)).round(2).tolist())

pair_diff = 100 * (test_nmae[:, 2] - test_nmae[:, best_baseline])
site_indices = [np.flatnonzero([row[0] == site for row in results]) for site in sites]
rng = np.random.default_rng(663)
samples = np.concatenate([rng.choice(group, (2000, len(group)), replace=True) for group in site_indices if len(group)], axis=1)
interval = np.quantile(np.median(pair_diff[samples], axis=1), [.025, .975])
print(f'Ridge wins against validation-selected baseline ({methods[best_baseline]}) on {int((pair_diff < 0).sum())}/{len(pair_diff)} test buildings.')
print(f'Median paired difference in NMAE (ridge minus baseline): {np.median(pair_diff):+.2f} pp; 95% site-stratified building bootstrap [{interval[0]:+.2f}, {interval[1]:+.2f}] pp.')

        Method  2016 validation median NMAE %  2017 test hourly MAE kWh  2017 test median building NMAE %
Copy yesterday                         11.400                     9.448                            12.470
Copy last week                         14.147                    11.634                            15.852
  Pooled ridge                         10.397                     8.843                            11.325
Site median building NMAE (%):
Bear 18 [11.58, 11.63, 10.49]
Fox 27 [10.43, 9.41, 8.2]
Panther 21 [10.49, 14.16, 9.67]
Rat 52 [14.38, 20.79, 14.09]
Weekday 118 [11.95, 15.64, 10.73]
Weekend 118 [14.21, 16.49, 12.39]
Ridge wins against validation-selected baseline (Copy yesterday) on 80/118 test buildings.
Median paired difference in NMAE (ridge minus baseline): -0.81 pp; 95% site-stratified building bootstrap [-1.37, -0.23] pp.


NMAE is absolute error divided by actual metered kWh, summed over the test hours of each building. The median gives each building equal weight in the summary. Hourly MAE is pooled across observations, so large loads and buildings with more valid windows can have more influence. The paired interval resamples buildings within the four observed sites. Results apply to the cleaned, eligible source readings and buildings at these sites. Evaluating transfers to a new site or IIT Kanpur would need a separate test. The default in the app comes from 2016 validation, independent of the 2017 test scores.

## What the experiment found

The 2017 test contains 118 buildings across four sites and 37,808 forecast days. Ridge regression reduced the median building normalized error to 11.33% from 12.47% for copying yesterday and 15.85% for copying last week. Copying yesterday was the stronger baseline on validation and won against ridge on some buildings. Ridge had lower normalized error on 80 of 118 test buildings. The median paired difference versus copying yesterday was −0.81 percentage points, with a 95% site-stratified building bootstrap interval of −1.37 to −0.23.

The app defaults to ridge based on validation and shows all three forecasts. The same public source supplied four illustrative holdout cases. Cases are selected by building ID hash and calendar order without using their forecast errors. The difference between the 639 raw building columns and the 611 published BuildingsBench count reflects separate cohort definitions. This study applies its own filtering and split and does not claim to reproduce the BuildingsBench cohort exactly.